# Data Preparation

### Purpose
Purpose of this notebook is to get the Mindware data (signals & events) ready for further processing.

### Data model
Each recording is a parent-child dyad. The raw data is organised as one file per subject **and** per event, using the filename convention `{subject_id}_{event_name}_{raw|event}.txt` (e.g. `T001_Baseline_raw.txt` and `T001_Baseline_event.txt`).

- There are 5 events: `Baseline`, `DCP`, `IDP`, `NDCP`, `Recovery`.
- Each event has a `*_raw.txt` file (signal) and a `*_event.txt` file (markers).
- The raw file contains both members of the dyad: `TECH-CHILD_Bio`/`TECH-CHILD_GSC` (child) and `TECH-PARENT_Bio`/`TECH-PARENT_GSC` (parent). The column order can differ between subjects, so columns are selected by name.
- The event file lists the marker times; the `Start` marker is the event onset and the `End` marker is the event offset. Every event's raw signal starts at `Time (s) = 0`.

### Approach
1. Discover all subjects from the `*_raw.txt` filenames.
2. For each subject and each event, read the raw signal and the event markers.
3. Split the signal into a **child** stream and a **parent** stream, renaming their `Bio`/`GSC` columns to the standard `MWMOBILEJ_Bio`/`MWMOBILEJ_GSC` used downstream.
4. Mark the onset (`Start`) and offset (`End`) rows for each event so the downstream segmentation can slice the task window.
5. Concatenate all events per role and export **one child file and one parent file per subject**.

### Input / Output

- Input `~/data/Sample TECH` (raw + event files in a single folder)
- Output `~/data/interim/signals` (one `{subject}_{role}_signal_events.csv` per subject and role) & `~/data/interim/events` (event markers per subject and role)


### Imports

In [1]:
# fmt: off
from pathlib import Path
from typing import Union, List, Dict
import sys
import pandas as pd 
import importlib
sys.path.append(str(Path().cwd().parent/"src"))
import ecg_utils.data_utils as data_utils
import ecg_utils.parameters as parameters
import ecg_utils.common as common
import numpy as np
importlib.reload(data_utils)
importlib.reload(parameters)
importlib.reload(common)
# fmt:on


<module 'ecg_utils.common' from 'C:\\Work\\Misc\\Yared\\neurophysiological_profiles\\src\\ecg_utils\\common.py'>

## Parameters

In [2]:
WORKING_DIR = Path().cwd()
ROOT_DIR = WORKING_DIR.parent

DATA_DIR = ROOT_DIR / 'data'

# In the new data model the raw signal files and the event files live together in a single folder.
RAW_DATA_DIR = DATA_DIR / 'Sample TECH'

INTERIM_SIGNAL_DATA_DIR = DATA_DIR / 'interim' / 'signals'
INTERIM_EVENT_DATA_DIR = DATA_DIR / 'interim' / 'events'

INTERIM_SIGNAL_DATA_DIR.mkdir(exist_ok=True, parents=True)
INTERIM_EVENT_DATA_DIR.mkdir(exist_ok=True, parents=True)

# Map each dyad role to its Bio (ECG) and GSC (EDA) column names in the raw files.
# The Bio/GSC columns are renamed to the standard names expected by the downstream notebooks.
ROLES = {
    "child": {"TECH-CHILD_Bio": "MWMOBILEJ_Bio", "TECH-CHILD_GSC": "MWMOBILEJ_GSC"},
    "parent": {"TECH-PARENT_Bio": "MWMOBILEJ_Bio", "TECH-PARENT_GSC": "MWMOBILEJ_GSC"},
}

# The event names to look for, taken from the segmentation config so this notebook stays in sync with the pipeline.
EXPECTED_EVENTS = [seg["event_name"] for seg in parameters.base_params["segmentation"].values()]

# Fixed analysis-window length per event: each segment runs from its Start marker to Start + duration.
# The 'End' marker in the event files is ignored.
SEGMENT_DURATIONS = {seg["event_name"]: seg["duration_seconds"] for seg in parameters.base_params["segmentation"].values()}

# Load and prepare raw data

Each subject has one `*_raw.txt` and one `*_event.txt` file per event (`Baseline`, `DCP`, `IDP`, `NDCP`, `Recovery`). The raw files hold both the child and the parent signals, so each subject is split into a separate child stream and parent stream.

In [3]:
sampling_frequency = parameters.base_params['general'].get("sampling_frequency")

In [4]:
"""
Get all unique subject ids from the raw filenames (the subject id is the first token, e.g. 'T001' in 'T001_Baseline_raw.txt')
"""
raw_filepaths = list(RAW_DATA_DIR.glob('*_raw.txt'))
all_subject_ids = sorted({flp.stem.split("_")[0] for flp in raw_filepaths})
print(f"Found {len(all_subject_ids)} subjects: {all_subject_ids}")


Found 2 subjects: ['T001', 'T003']


In [5]:
"""
For each subject: read every event's raw + event file, split into a child and a parent stream,
mark the onset/offset of each event, concatenate the events, and export one file per role.

The segment window runs from the 'Start' marker to Start + a fixed duration (see SEGMENT_DURATIONS);
the 'End' marker in the event files is ignored. Statistics and any problems encountered are collected
in `segment_records` and `problems` for the report generated in the next cell.
"""
segment_records = []   # one row per (subject, event) describing the extracted window
problems = []          # human-readable descriptions of issues worth flagging to the researcher
exported_files = []    # interim signal files that were written

for subject_id in all_subject_ids:
    print(f"Processing subject {subject_id}")
    # if subject_id != "T001":
    #     continue

    # Accumulate the per-event dataframes for each role (child / parent)
    role_frames = {role: [] for role in ROLES}

    for event_name in EXPECTED_EVENTS:
        raw_filepath = RAW_DATA_DIR / f"{subject_id}_{event_name}_raw.txt"
        event_filepath = RAW_DATA_DIR / f"{subject_id}_{event_name}_event.txt"
        if not raw_filepath.exists() or not event_filepath.exists():
            print(f"  Skipping event '{event_name}': missing raw and/or event file")
            problems.append(f"{subject_id} / {event_name}: missing raw and/or event file - event skipped.")
            segment_records.append({
                "subject_id": subject_id, "event_name": event_name, "status": "missing_file",
                "start_time_s": np.nan, "requested_duration_s": SEGMENT_DURATIONS[event_name],
                "actual_duration_s": np.nan, "recording_length_s": np.nan, "truncated": np.nan,
            })
            continue

        # Read the raw signal (skip the 'Sample Rate:' line so the second line becomes the header)
        raw_df = pd.read_csv(raw_filepath, delimiter="\t", skiprows=1)
        raw_df["Time (s)"] = raw_df["Time (s)"].apply(lambda x: common.comma_str_2_float(x) if isinstance(x, str) else x)

        # Read the event markers and locate the onset ('Start') time. The 'End' marker is ignored.
        event_df = pd.read_csv(event_filepath, delimiter="\t")
        event_df["Time"] = event_df["Time"].apply(common.comma_str_2_float)
        start_times = event_df.loc[event_df["Name"] == "Start", "Time"]

        # The raw signal starts at t=0 and is sampled at a fixed rate, so a marker time maps to a row position.
        last_row = len(raw_df) - 1
        start_marker_present = not start_times.empty
        if start_marker_present:
            start_time = start_times.iloc[0]
            onset_row = int(round(start_time * sampling_frequency))
        else:
            print(f"  '{event_name}': no 'Start' marker found; using the first sample as onset")
            problems.append(f"{subject_id} / {event_name}: no 'Start' marker found - onset defaulted to the first sample (t=0).")
            start_time = 0.0
            onset_row = 0

        # Offset is a fixed window after the onset (the End marker is not used)
        requested_duration_s = SEGMENT_DURATIONS[event_name]
        onset_row = min(max(onset_row, 0), last_row)
        offset_row_requested = onset_row + int(round(requested_duration_s * sampling_frequency))
        offset_row = min(offset_row_requested, last_row)

        # Flag recordings that are too short to cover the full requested window
        truncated = offset_row_requested > last_row
        actual_duration_s = (offset_row - onset_row) / sampling_frequency
        recording_length_s = len(raw_df) / sampling_frequency
        if truncated:
            problems.append(
                f"{subject_id} / {event_name}: recording too short for the {requested_duration_s:.0f}s window - "
                f"only {actual_duration_s:.1f}s available after the Start marker; segment truncated."
            )
        if offset_row <= onset_row:
            problems.append(
                f"{subject_id} / {event_name}: no samples available after the Start marker - resulting segment is empty."
            )

        segment_records.append({
            "subject_id": subject_id, "event_name": event_name, "status": "ok",
            "start_time_s": round(start_time, 3), "requested_duration_s": requested_duration_s,
            "actual_duration_s": round(actual_duration_s, 3), "recording_length_s": round(recording_length_s, 3),
            "truncated": truncated,
        })

        # Build a separate stream for each role and mark the onset/offset rows
        for role, column_rename_map in ROLES.items():
            role_df = raw_df[["Time (s)", *column_rename_map.keys()]].rename(columns=column_rename_map).copy()
            role_df["event_name"] = np.nan
            role_df["on_offset"] = np.nan
            if 0 <= onset_row < len(role_df):
                role_df.loc[onset_row, ["event_name", "on_offset"]] = [event_name, "onset"]
            if 0 <= offset_row < len(role_df):
                role_df.loc[offset_row, ["event_name", "on_offset"]] = [event_name, "offset"]
            role_frames[role].append(role_df)

    # Concatenate all events per role, add metadata, and export one file per role
    for role, frames in role_frames.items():
        if not frames:
            print(f"  No usable data for role '{role}'; skipping export")
            problems.append(f"{subject_id} / {role}: no usable events - no interim file written.")
            continue
        role_subject_id = f"{subject_id}_{role}"
        signal_df = pd.concat(frames, ignore_index=True)
        signal_df = (
            signal_df
            .assign(subject_id=role_subject_id, role=role, row_index=range(len(signal_df)))
            .rename(columns={"Time (s)": "time_seconds_original_file"})
            .set_index("row_index", drop=True)
        )
        signal_df.index.name = "row_index"
        output_path = INTERIM_SIGNAL_DATA_DIR / f"{role_subject_id}_signal_events.csv"
        signal_df.to_csv(output_path, index=False)
        signal_df[~pd.isnull(signal_df["event_name"])].to_excel(INTERIM_EVENT_DATA_DIR / f"{role_subject_id}_events.xlsx", index=False)
        exported_files.append(output_path.name)

print(f"Done. Wrote {len(exported_files)} interim files with {len(problems)} problem(s) logged.")
    

Processing subject T001


C:\Users\natih\AppData\Local\Temp\ipykernel_31136\957755400.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Baseline' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  role_df.loc[onset_row, ["event_name", "on_offset"]] = [event_name, "onset"]
C:\Users\natih\AppData\Local\Temp\ipykernel_31136\957755400.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'onset' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  role_df.loc[onset_row, ["event_name", "on_offset"]] = [event_name, "onset"]
C:\Users\natih\AppData\Local\Temp\ipykernel_31136\957755400.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Baseline' has dtype incompatible with float64, please explici

Processing subject T003


C:\Users\natih\AppData\Local\Temp\ipykernel_31136\957755400.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Baseline' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  role_df.loc[onset_row, ["event_name", "on_offset"]] = [event_name, "onset"]
C:\Users\natih\AppData\Local\Temp\ipykernel_31136\957755400.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'onset' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  role_df.loc[onset_row, ["event_name", "on_offset"]] = [event_name, "onset"]
C:\Users\natih\AppData\Local\Temp\ipykernel_31136\957755400.py:88: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Baseline' has dtype incompatible with float64, please explici

Done. Wrote 4 interim files with 1 problem(s) logged.


# Data preparation report

Write a human-readable report summarising what was processed, the segmentation rules that were applied, and any problems the researcher should be aware of (missing files, missing markers, truncated segments). The report is saved to `~/reports/data_preparation_report.md`, and the per-segment details are also saved as a CSV alongside it.

In [6]:
from datetime import datetime

REPORTS_DIR = ROOT_DIR / 'reports'
REPORTS_DIR.mkdir(exist_ok=True, parents=True)
report_path = REPORTS_DIR / 'data_preparation_report.md'
records_csv_path = REPORTS_DIR / 'data_preparation_report.csv'


def _df_to_markdown(df: pd.DataFrame) -> str:
    """Render a DataFrame as a Markdown table without requiring the optional 'tabulate' dependency."""
    if df.empty:
        return "_No segments were processed._"
    columns = list(df.columns)
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = ["| " + " | ".join(str(value) for value in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join([header, separator] + rows)


records_df = pd.DataFrame(segment_records)
records_df.to_csv(records_csv_path, index=False)

# Headline numbers
n_subjects = len(all_subject_ids)
n_interim_files = len(exported_files)
n_events_expected = n_subjects * len(EXPECTED_EVENTS)
n_events_processed = int((records_df["status"] == "ok").sum()) if not records_df.empty else 0
n_missing = int((records_df["status"] == "missing_file").sum()) if not records_df.empty else 0
ok_records = records_df[records_df["status"] == "ok"] if not records_df.empty else records_df
n_truncated = int(ok_records["truncated"].fillna(False).sum()) if not ok_records.empty else 0

duration_rule = ", ".join(f"`{event}` = {dur}s" for event, dur in SEGMENT_DURATIONS.items())

lines = [
    "# Data Preparation Report",
    "",
    f"_Generated: {datetime.now():%Y-%m-%d %H:%M:%S}_",
    "",
    "## Processing rules",
    "- Each subject (parent-child dyad) is split into a **child** and a **parent** stream, each exported as a separate interim file.",
    "- Signal columns are selected by name (`TECH-CHILD_*` / `TECH-PARENT_*`), so a differing column order between subjects is handled automatically.",
    "- Each segment starts at its `Start` marker; the `End` marker in the event files is **ignored**.",
    f"- Fixed analysis-window length per event: {duration_rule}.",
    f"- Sampling frequency: {sampling_frequency} Hz.",
    "",
    "## Summary",
    f"- Subjects found: **{n_subjects}** ({', '.join(all_subject_ids) if all_subject_ids else 'none'})",
    f"- Interim files written: **{n_interim_files}** (a child and a parent file per subject)",
    f"- Event-segments processed: **{n_events_processed} / {n_events_expected}**",
    f"- Missing event/raw files: **{n_missing}**",
    f"- Truncated segments (recording shorter than the requested window): **{n_truncated}**",
    "",
    "## Outputs",
    f"- Signal files: `{INTERIM_SIGNAL_DATA_DIR}` as `{{subject}}_{{role}}_signal_events.csv` (e.g. `T001_child_signal_events.csv`).",
    f"- Event markers: `{INTERIM_EVENT_DATA_DIR}` as `{{subject}}_{{role}}_events.xlsx`.",
    "- In the outputs, `subject_id` is set to `{subject}_{role}` (e.g. `T001_child`) so downstream per-subject processing keeps child and parent separate.",
    "",
    "## Segment details",
    "",
    _df_to_markdown(records_df),
    "",
    "## Problems and notes",
]

if problems:
    lines.extend(f"- {problem}" for problem in problems)
else:
    lines.append("- No problems encountered.")
lines.append("")

report_text = "\n".join(lines)
report_path.write_text(report_text, encoding="utf-8")
print(f"Report written to {report_path}")
print(f"Segment details written to {records_csv_path}")
print()
print(report_text)

Report written to C:\Work\Misc\Yared\neurophysiological_profiles\reports\data_preparation_report.md
Segment details written to C:\Work\Misc\Yared\neurophysiological_profiles\reports\data_preparation_report.csv

# Data Preparation Report

_Generated: 2026-07-29 19:20:20_

## Processing rules
- Each subject (parent-child dyad) is split into a **child** and a **parent** stream, each exported as a separate interim file.
- Signal columns are selected by name (`TECH-CHILD_*` / `TECH-PARENT_*`), so a differing column order between subjects is handled automatically.
- Each segment starts at its `Start` marker; the `End` marker in the event files is **ignored**.
- Fixed analysis-window length per event: `Baseline` = 300s, `DCP` = 600s, `IDP` = 600s, `NDCP` = 600s, `Recovery` = 300s.
- Sampling frequency: 500 Hz.

## Summary
- Subjects found: **2** (T001, T003)
- Interim files written: **4** (a child and a parent file per subject)
- Event-segments processed: **10 / 10**
- Missing event/raw files